**Dlaczego wynik jest prawie zawsze inny niż 0? Jaki rodzaj błędu występuje?**
Wynika to ze wspóbieżnego dostępu dwóch wątków do tej samej zmiennej counter. Jest to błąd nazywany race condition.

**Które fragmenty kodu w funkcjach incrementer i decrementer powinny być chronione?**
Chronione powinny być oprecje modyfikujące współdzieloną zmienną counter, cyzli insturkcje: counter += 1 i counter -= 1. 

**Czy wystarczy użyć jednej, wspólnej blokady dla obu funkcji?**
Tak, wystarczy jedna wspólna blokada (threading.Lock()). Gwarantuje to, że w danym momencie tylko jede wątek może modyfikować licznik.

In [1]:
import threading
import time
import threading

counter = 0
lock = threading.Lock()

def incrementer():
    global counter
    for _ in range(100_000):
        with lock:
            counter += 1

def decrementer():
    global counter
    for _ in range(100_000):
        with lock:
            counter -= 1

t1 = threading.Thread(target=incrementer)
t2 = threading.Thread(target=decrementer)

t1.start()
t2.start()

t1.join()
t2.join()

print(f"Końcowa wartość licznika: {counter}")

Końcowa wartość licznika: 0


**W którym dokładnie momencie występuje race condition?**
Race condition występuje pomiędzy sprawdzeniem while tasks: a wykonaniem tasks.pop(0).

**Co się stanie, jeśli oba wątki sprawdzą while tasks: w tym samym czasie, gdy na liście zostaje tylko jedno zadanie?**
Jeśli dwa wątki jednocześnie sprawdzą, że lista nie jest pusta, a pozostanie tylko jedno zadanie, jeden wątek pobierze je poprawnie, a drugi spróbuje pobrać element z pustej listy i otrzyma IndexError.

**Jaką operację na liście tasks trzeba chronić blokadą?** Blokadą należy chronić operację sprawdzenia, czy lista jest pusta, oraz pobrania elementu (tasks.pop(0)). Operacje te powinny stanowić jedną sekcję krytyczną.

**Czy blokada powinna być wewnątrz pętli while, czy na zewnątrz?** Blokada powinna znajdować się wewnątrz pętli, aby była zajęta tylko podczas pobierania zadania. Umieszczenie jej na zewnątrz pętli spowodowałoby, że jeden wątek wykona wszystkie zadania samodzielnie, blokując pozostałe.

In [2]:
import threading
import time

tasks = [f"Zadanie {i}" for i in range(10)]
lock = threading.Lock()

def worker(name):
    while True:

        with lock:
            if not tasks:
                break

            # Pobierz pierwsze zadanie z listy
            task = tasks.pop(0)

        print(f"Wątek {name} pobiera: {task}")

        # Symuluj pracę nad zadaniem
        time.sleep(0.01)

t1 = threading.Thread(target=worker, args=("A",))
t2 = threading.Thread(target=worker, args=("B",))

t1.start()
t2.start()

t1.join()
t2.join()

print("Wszystkie zadania ukończone.")

Wątek A pobiera: Zadanie 0
Wątek B pobiera: Zadanie 1
Wątek A pobiera: Zadanie 2
Wątek B pobiera: Zadanie 3
Wątek A pobiera: Zadanie 4
Wątek B pobiera: Zadanie 5
Wątek A pobiera: Zadanie 6
Wątek B pobiera: Zadanie 7
Wątek A pobiera: Zadanie 8
Wątek B pobiera: Zadanie 9
Wszystkie zadania ukończone.


**Opis krok po kroku, jak dochodzi do sytuacji, w której oba wątki czekają na siebie nawzajem.**
Wątek 1 zdobywa blokadę A i czeka na blokadę B. Jednocześnie wątek 2 zdobywa blokadę B i czeka na blokadę A. Każdy wątek czeka na zasób trzymany przez drugi wątek, więc żaden nie może kontynuować działania.

**Co to jest zakleszczenie (deadlock)?**
Deadlock to sytuacja, w której dwa lub więcej wątków czeka na siebie nawzajem w nieskończoność, przez co program przestaje wykonywać dalsze instrukcje.

**Jaką prostą zasadę można wprowadzić, aby uniknąć tego typu problemów z wieloma
blokadami?**
Oba wątki powinny pobierać najpierw blokadę A, a następnie blokadę B.

In [3]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()

def thread_1():
    print("Wątek 1: Próbuję zdobyć blokadę A...")
    with lock_a:
        print("Wątek 1: Zdobyłem blokadę A. Czekam 1s...")
        time.sleep(1)

        print("Wątek 1: Próbuję zdobyć blokadę B...")
        with lock_b:
            print("Wątek 1: Zdobyłem obie blokady!")

def thread_2():
    print("Wątek 2: Próbuję zdobyć blokadę A...")
    with lock_a:
        print("Wątek 2: Zdobyłem blokadę A. Czekam 1s...")
        time.sleep(1)

        print("Wątek 2: Próbuję zdobyć blokadę B...")
        with lock_b:
            print("Wątek 2: Zdobyłem obie blokady!")

t1 = threading.Thread(target=thread_1)
t2 = threading.Thread(target=thread_2)

t1.start()
t2.start()

t1.join()
t2.join()

print("Program zakończony pomyślnie.")

Wątek 1: Próbuję zdobyć blokadę A...
Wątek 1: Zdobyłem blokadę A. Czekam 1s...
Wątek 2: Próbuję zdobyć blokadę A...
Wątek 1: Próbuję zdobyć blokadę B...
Wątek 1: Zdobyłem obie blokady!
Wątek 2: Zdobyłem blokadę A. Czekam 1s...
Wątek 2: Próbuję zdobyć blokadę B...
Wątek 2: Zdobyłem obie blokady!
Program zakończony pomyślnie.
